# Notebook 16 — No-reference image quality (NIQE)

**Not yet executed.** Quantifies no-reference image *naturalness*, not face fidelity, emotion validity, identity change or clinical suitability. Lower NIQE is typically better. NIQE is sensitive to face crops, grayscale images and unusual exposures; report source grayscale and quality strata separately.

- Frozen inputs: N07 full 600 generated images, 120 originals; N12 V3 only supplies privacy scores for downstream contextual analysis.
- No new diffusion generation, no face images or per-image scores in public GitHub.
- Preflight validates all 600 expected PNGs and paired source keys.
- Saves private row-level NIQE and source-cluster bootstrap aggregates to a new Drive run.
- NIQE from `pyiqa` is a pretrained reference-free quality evaluator; requires downloaded pretrained NIQE model resources, therefore the first runtime needs internet. Do not infer image quality from missing metric outputs.


In [ ]:
!pip -q install "pyiqa==0.1.13" "pandas>=2.2,<3" "Pillow>=10,<12"
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import numpy as np, pandas as pd, torch, pyiqa, json
from PIL import Image
from datetime import datetime, timezone
ROOT=Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07=ROOT/'20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
assert N07.is_dir()
raw=pd.read_csv(N07/'raw_results.csv',dtype={'source_id':'string','condition':'string'})
sources=pd.read_csv(N07/'selected_sources.csv',dtype={'source_id':'string'})
assert len(raw)==600 and len(sources)==120 and sources.source_id.nunique()==120
keys=['source_id','condition','seed']
assert not raw.duplicated(keys).any()
assert set(raw.generation_status)=={'success'}
# Resolve paths relative to the frozen run, not an arbitrary local copy.
def resolve_image(old_path):
    p=Path(str(old_path))
    q=N07/'images'/p.name
    assert q.is_file(),f'Missing generated image: {q}'
    return q
raw['generated_file']=raw.output_path.map(resolve_image)
src_dir=N07/'sources'
assert src_dir.is_dir()
sources['original_file']=sources.source_id.map(lambda x:src_dir/(str(x)+'.png'))
assert sources.original_file.map(lambda p:p.is_file()).all()
print('Preflight: 600 outputs, 120 originals and all frozen files present.')


In [ ]:
# Metric computation on original 512x512 images, with RGB conversion documented.
# NIQE model and device intentionally pinned in the report.
metric=pyiqa.create_metric('niqe',device='cpu')
metric.eval()
def niqe(path):
    # pyiqa inference expects normalized tensor N,C,H,W (RGB).
    with Image.open(path) as img:
        rgb=np.asarray(img.convert('RGB'),dtype=np.float32)/255.
    tensor=torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0)
    with torch.no_grad():
        score=float(metric(tensor).detach().cpu().item())
    assert np.isfinite(score),f'Non-finite NIQE: {path}'
    return score
rows=[]
for i,r in enumerate(raw.itertuples(index=False),1):
    rows.append({'source_id':r.source_id,'condition':r.condition,'seed':int(r.seed),'niqe':niqe(r.generated_file)})
    if i%50==0:print(f'NIQE processed {i}/600')
quality=pd.DataFrame(rows)
ref=pd.DataFrame([{'source_id':r.source_id,'original_niqe':niqe(r.original_file)} for r in sources.itertuples(index=False)])
quality=quality.merge(ref,on='source_id',validate='many_to_one')
quality['niqe_change_vs_original']=quality.niqe-quality.original_niqe
assert len(quality)==600 and quality[keys].apply(tuple,axis=1).is_unique


In [ ]:
# Grouped descriptives, source-cluster bootstrap CIs; no independence assumption across repeated seeds.
rng=np.random.default_rng(16016)
def source_bootstrap_ci(frame,col,B=2000):
    clusters=[f[col].to_numpy(dtype=float) for _,f in frame.groupby('source_id')]
    res=[]
    for _ in range(B):
        idx=rng.integers(0,len(clusters),len(clusters))
        res.append(np.concatenate([clusters[i] for i in idx]).mean())
    return np.quantile(res,[.025,.975]).tolist()
summary=[]
for name,group in quality.groupby('condition',sort=True):
    for col in ['niqe','niqe_change_vs_original']:
        low,high=source_bootstrap_ci(group,col)
        summary.append({'condition':name,'metric':col,'n_pairs':len(group),'n_sources':group.source_id.nunique(),'mean':group[col].mean(),'ci95_low':low,'ci95_high':high})
summary=pd.DataFrame(summary)
out=ROOT/('nb16_no_reference_niqe_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
out.mkdir(parents=True,exist_ok=False)
quality.to_csv(out/'niqe_per_image_PRIVATE.csv',index=False)
summary.to_csv(out/'niqe_aggregate.csv',index=False)
status={'notebook':'16','status':'EXECUTED','niqe_implementation':'pyiqa 0.1.13','metric':'NIQE','direction':'lower_is_better_typically','n_pairs':600,'n_sources':120,'interpretation':'Natural-image statistics only. Not face-identity, emotion, or absolute clinical image quality.','limitations':['face content sensitivity','grayscale/RGB-converted images','source photo compression','illumination','pose','selection effects'],'publication':'Only aggregate NIQE tables after scientific review; do not commit per-image csv.'}
(out/'evidence_status.json').write_text(json.dumps(status,indent=2))
display(summary.round(4))
print('Saved private run:',out)
